In [1]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [4]:
#2. IMPORT ALL FOUR DATASETS


movies = pd.read_csv("cleaned_movies.csv")
ratings = pd.read_csv("cleaned_ratings.csv")
tags = pd.read_csv("cleaned_tags.csv")
links = pd.read_csv("cleaned_links.csv")

print("\nDATASETS IMPORTED SUCCESSFULLY")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)
print("Links:", links.shape)




DATASETS IMPORTED SUCCESSFULLY
Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)
Links: (9742, 3)


In [6]:
# CHAPTER A: MEASURES OF CENTRAL TENDENCY

print("MEASURES OF CENTRAL TENDENCY")

# Define rating_values from the ratings DataFrame
rating_values = ratings["rating"]

# 1. Mean
mean_rating = rating_values.mean()
print("Mean:", mean_rating)

# 2. Median
median_rating = rating_values.median()
print("Median:", median_rating)

# 3. Mode
mode_rating = rating_values.mode()
print("Mode:", mode_rating.tolist())

# 4. Weighted Mean
# Average movie ratings weighted by number of ratings

movie_stats = ratings.groupby("movieId")["rating"].agg(
    mean_rating="mean",
    rating_count="count",
    std_rating="std"
).reset_index()

weighted_mean = np.average(
    movie_stats["mean_rating"],
    weights=movie_stats["rating_count"]
)

print("Weighted Mean:", weighted_mean)

# 5. Descriptive Statistical Summary
print("\nDescriptive Summary:")
print(rating_values.describe())

MEASURES OF CENTRAL TENDENCY
Mean: 3.501556983616962
Median: 3.5
Mode: [4.0]
Weighted Mean: 3.501556983616962

Descriptive Summary:
count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


In [7]:
print("MEASURES OF DISPERSION")

# 1. Range
data_range = rating_values.max() - rating_values.min()
print("Range:", data_range)

# 2. Variance
variance = rating_values.var()
print("Variance:", variance)

# 3. Standard Deviation
std_deviation = rating_values.std()
print("Standard Deviation:", std_deviation)

# 4. Interquartile Range (IQR)
Q1 = rating_values.quantile(0.25)
Q3 = rating_values.quantile(0.75)

IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

# 5. Mean Absolute Deviation
MAD = (rating_values - rating_values.mean()).abs().mean()
print("Mean Absolute Deviation:", MAD)

# 6. Coefficient of Variation
CV = (std_deviation / mean_rating) * 100
print("Coefficient of Variation:", CV, "%")


MEASURES OF DISPERSION
Range: 4.5
Variance: 1.08686721429614
Standard Deviation: 1.0425292390605359
Q1: 3.0
Q3: 4.0
IQR: 1.0
Mean Absolute Deviation: 0.8271026565093413
Coefficient of Variation: 29.773304959431123 %


In [8]:
print("MEASURES OF POSITION AND DISTRIBUTION")

# 1. Quartiles
print("\nQuartiles:")
print(rating_values.quantile([0.25, 0.50, 0.75]))

# 2. Deciles
print("\nDeciles:")
print(rating_values.quantile(
    [0.1, 0.2, 0.3, 0.4, 0.5,
     0.6, 0.7, 0.8, 0.9]
))

# 3. Percentiles
print("\nPercentiles:")
print("25th Percentile:", rating_values.quantile(0.25))
print("50th Percentile:", rating_values.quantile(0.50))
print("75th Percentile:", rating_values.quantile(0.75))
print("90th Percentile:", rating_values.quantile(0.90))

# 4. Skewness
skewness = rating_values.skew()
print("\nSkewness:", skewness)

# 5. Kurtosis
kurtosis = rating_values.kurt()
print("Kurtosis:", kurtosis)


MEASURES OF POSITION AND DISTRIBUTION

Quartiles:
0.25    3.0
0.50    3.5
0.75    4.0
Name: rating, dtype: float64

Deciles:
0.1    2.0
0.2    3.0
0.3    3.0
0.4    3.5
0.5    3.5
0.6    4.0
0.7    4.0
0.8    4.5
0.9    5.0
Name: rating, dtype: float64

Percentiles:
25th Percentile: 3.0
50th Percentile: 3.5
75th Percentile: 4.0
90th Percentile: 5.0

Skewness: -0.6371993892529213
Kurtosis: 0.12331407123922533


In [9]:
print("COVARIANCE AND CORRELATION")

# Movie-wise average rating and number of ratings
movie_stats = ratings.groupby("movieId")["rating"].agg(
    mean_rating="mean",
    rating_count="count"
).reset_index()

# 1. Covariance
covariance = movie_stats["mean_rating"].cov(
    movie_stats["rating_count"]
)

print("Covariance:", covariance)

# 2. Pearson Correlation
pearson_corr = movie_stats["mean_rating"].corr(
    movie_stats["rating_count"],
    method="pearson"
)

print("Pearson Correlation:", pearson_corr)

# 3. Spearman Rank Correlation
spearman_corr = movie_stats["mean_rating"].corr(
    movie_stats["rating_count"],
    method="spearman"
)

print("Spearman Correlation:", spearman_corr)

# 4. Covariance Matrix
print("\nCovariance Matrix:")
print(movie_stats[["mean_rating", "rating_count"]].cov())

# 5. Correlation Matrix
print("\nCorrelation Matrix:")
print(movie_stats[["mean_rating", "rating_count"]].corr())


COVARIANCE AND CORRELATION
Covariance: 2.4797660970031594
Pearson Correlation: 0.12725857359560636
Spearman Correlation: 0.03958153228619635

Covariance Matrix:
              mean_rating  rating_count
mean_rating      0.756680      2.479766
rating_count     2.479766    501.805016

Correlation Matrix:
              mean_rating  rating_count
mean_rating      1.000000      0.127259
rating_count     0.127259      1.000000


In [10]:
print("VECTOR OPERATIONS")

# Select movies with at least two ratings
vector_data = movie_stats.dropna(subset=["mean_rating"])

vector_data = vector_data.sort_values(
    "rating_count", ascending=False
).head(10)

# Each movie is represented by:
# [Average Rating, Number of Ratings]

movie_vectors = vector_data[
    ["mean_rating", "rating_count"]
].to_numpy()

# Select two movie vectors
vector1 = movie_vectors[0]
vector2 = movie_vectors[1]

print("Vector 1:", vector1)
print("Vector 2:", vector2)

# 1. Vector Addition
print("Vector Addition:", vector1 + vector2)

# 2. Vector Subtraction
print("Vector Subtraction:", vector1 - vector2)

# 3. Scalar Multiplication
print("Scalar Multiplication:", 2 * vector1)

# 4. Dot Product
print("Dot Product:", np.dot(vector1, vector2))

# 5. Magnitude of Vector
print("Magnitude of Vector 1:", np.linalg.norm(vector1))

# 6. Mean of Movie Feature Vectors
print("Mean Vector:", np.mean(movie_vectors, axis=0))



VECTOR OPERATIONS
Vector 1: [  4.16413374 329.        ]
Vector 2: [  4.42902208 317.        ]
Vector Addition: [  8.59315582 646.        ]
Vector Subtraction: [-0.26488834 12.        ]
Scalar Multiplication: [  8.32826748 658.        ]
Dot Product: 104311.44304028075
Magnitude of Vector 1: 329.0263515431446
Mean Vector: [  4.1352664 268.       ]


In [11]:
print("MATRIX OPERATIONS")

# Create a matrix using movie statistics
# Rows represent movies
# Columns represent average rating and rating count

matrix = movie_vectors

print("\nOriginal Matrix:")
print(matrix)

# 1. Matrix Addition
print("\nMatrix Addition:")
print(matrix + matrix)

# 2. Matrix Subtraction
print("\nMatrix Subtraction:")
print(matrix - matrix)

# 3. Scalar Multiplication
print("\nScalar Multiplication:")
print(2 * matrix)

# 4. Transpose
print("\nTranspose:")
print(matrix.T)

# 5. Matrix Multiplication
print("\nMatrix Multiplication:")
print(np.dot(matrix.T, matrix))

# 6. Matrix Mean
print("\nMatrix Mean:")
print(np.mean(matrix))



MATRIX OPERATIONS

Original Matrix:
[[  4.16413374 329.        ]
 [  4.42902208 317.        ]
 [  4.1970684  307.        ]
 [  4.16129032 279.        ]
 [  4.19244604 278.        ]
 [  4.2310757  251.        ]
 [  3.75       238.        ]
 [  4.03164557 237.        ]
 [  3.97098214 224.        ]
 [  4.225      220.        ]]

Matrix Addition:
[[  8.32826748 658.        ]
 [  8.85804416 634.        ]
 [  8.39413681 614.        ]
 [  8.32258065 558.        ]
 [  8.38489209 556.        ]
 [  8.46215139 502.        ]
 [  7.5        476.        ]
 [  8.06329114 474.        ]
 [  7.94196429 448.        ]
 [  8.45       440.        ]]

Matrix Subtraction:
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]

Scalar Multiplication:
[[  8.32826748 658.        ]
 [  8.85804416 634.        ]
 [  8.39413681 614.        ]
 [  8.32258065 558.        ]
 [  8.38489209 556.        ]
 [  8.46215139 502.        ]
 [  7.5        476.        ]
 [  8.06329114 474.      

In [12]:
print("NORMALIZATION AND STANDARDIZATION")

# Convert ratings into a 2D array
rating_array = rating_values.to_numpy().reshape(-1, 1)

# 1. Min-Max Normalization
minmax = MinMaxScaler()

normalized_data = minmax.fit_transform(rating_array)

print("\nMin-Max Normalized Ratings:")
print(normalized_data[:10])

# 2. Z-Score Standardization
standard_scaler = StandardScaler()

standardized_data = standard_scaler.fit_transform(rating_array)

print("\nStandardized Ratings:")
print(standardized_data[:10])

# 3. Mean Normalization
mean_normalized = (
    rating_values - rating_values.mean()
) / (rating_values.max() - rating_values.min())

print("\nMean Normalized Ratings:")
print(mean_normalized.head(10))





print("CINEMATRIX STATISTICS ANALYSIS COMPLETED")


NORMALIZATION AND STANDARDIZATION

Min-Max Normalized Ratings:
[[0.77777778]
 [0.77777778]
 [0.77777778]
 [1.        ]
 [1.        ]
 [0.55555556]
 [1.        ]
 [0.77777778]
 [1.        ]
 [1.        ]]

Standardized Ratings:
[[ 0.47811176]
 [ 0.47811176]
 [ 0.47811176]
 [ 1.43732223]
 [ 1.43732223]
 [-0.48109871]
 [ 1.43732223]
 [ 0.47811176]
 [ 1.43732223]
 [ 1.43732223]]

Mean Normalized Ratings:
0    0.110765
1    0.110765
2    0.110765
3    0.332987
4    0.332987
5   -0.111457
6    0.332987
7    0.110765
8    0.332987
9    0.332987
Name: rating, dtype: float64
CINEMATRIX STATISTICS ANALYSIS COMPLETED
